In [1]:
!pip install -qq faiss-cpu      # Vector search library
!pip install -qq transformers   # HuggingFace models
!pip install -qq pandas         # Đọc CSV
!pip install -qq numpy          # Tính toán số học
!pip install -qq scikit-learn   # train_test_split, LabelEncoder
!pip install -qq tqdm           # Progress bar

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 71.5 MB/s eta 0:00:00


In [2]:
# Tải file cls_spam_text.csv từ Google Drive
# File ID lấy từ URL: /file/d/1yuAdAo.../view
!gdown 1yuAdAo3Gex-HsE1hB3Fa1hL5ucew4Jtb

Downloading...
From: https://drive.google.com/uc?id=1yuAdAo3Gex-HsE1hB3Fa1hL5ucew4Jtb
To: /content/cls_spam_text.csv
100% 486k/486k [00:00<00:00, 122MB/s]


In [3]:
# Nếu bị rate limit, dùng cách thủ công:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
import faiss
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm

In [5]:
# Đọc CSV
DATASET_PATH = 'cls_spam_text.csv'
df = pd.read_csv(DATASET_PATH)


In [6]:
# Xem vài dòng đầu
print(df.head())
print(df.shape)          # (5574, 2)
print(df['Category'].value_counts())

  Category                                            Message
0      ham  Go until jurong point, crazy.. Available only ...
1      ham                      Ok lar... Joking wif u oni...
2     spam  Free entry in 2 a wkly comp to win FA Cup fina...
3      ham  U dun say so early hor... U c already then say...
4      ham  Nah I don't think he goes to usf, he lives aro...
(5572, 2)
Category
ham     4825
spam     747
Name: count, dtype: int64


In [7]:
# Tách thành 2 lists
messages = df['Message'].values.tolist()  # ['Go until jurong...', ...]
labels = df['Category'].values.tolist()   # ['ham', 'spam', 'ham', ...]

In [8]:
MODEL_NAME = 'intfloat/multilingual-e5-base'

In [9]:
# AutoTokenizer: tự detect đúng tokenizer cho model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

In [10]:
# AutoModel: load weights đã pretrained
model = AutoModel.from_pretrained(MODEL_NAME)

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [11]:
# Set device: dùng GPU nếu có, ngược lại CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

In [12]:
# eval() = inference mode: tắt dropout, batch norm dùng running stats
# QUAN TRỌNG: phải gọi .eval() trước khi inference!
model.eval()

print(f'Using device: {device}')  # cuda:0 nếu có GPU

Using device: cuda


In [17]:
def average_pool(last_hidden_states, attention_mask):
    # Step 1: Mask padding tokens về 0
    # attention_mask = 1 cho real token, 0 cho [PAD]
    # ~attention_mask = lấy ngược: 1 cho padding
    # masked_fill(..., 0.0) → set padding hidden states = 0
    last_hidden = last_hidden_states.masked_fill(
        ~attention_mask[..., None].bool(), 0.0
    )
    # Shape: [batch_size, seq_len, 768]
    # [PAD] tokens giờ = zero vector, không ảnh hưởng sum

    # Step 2: Sum tất cả real token embeddings
    # last_hidden.sum(dim=1) → [batch_size, 768]

    # Step 3: Chia cho số real tokens để lấy trung bình
    # attention_mask.sum(dim=1) = số real token mỗi câu
    # [batch_size, None] để broadcast chia đúng
    return last_hidden.sum(dim=1) / attention_mask.sum(dim=1)[..., None]

In [18]:
def encode_text(batch_texts, model, tokenizer, device):
    # Tokenize: chuyển list of strings thành tensors
    batch_dict = tokenizer(
        batch_texts,
        max_length=512,      # truncate nếu dài hơn
        padding=True,        # pad ngắn hơn về cùng độ dài
        truncation=True,     # cắt nếu > max_length
        return_tensors='pt'  # trả về PyTorch tensors
    )
    # batch_dict = {input_ids, attention_mask, token_type_ids}

    # Move tensors lên đúng device (CPU/GPU)
    batch_dict = {k: v.to(device) for k, v in batch_dict.items()}

    # torch.no_grad(): tắt autograd để tiết kiệm memory/time
    # Không cần gradient khi inference
    with torch.no_grad():
        outputs = model(**batch_dict)
        # outputs.last_hidden_state: [batch, seq_len, 768]

        # Average pool → [batch, 768]
        batch_embeddings = average_pool(
            outputs.last_hidden_state,
            batch_dict['attention_mask']
        )

        # L2 normalize → đơn vị hóa, mỗi vector có độ dài = 1
        # p=2: L2 norm, dim=1: normalize theo chiều embedding
        batch_embeddings = F.normalize(batch_embeddings, p=2, dim=1)

    # .cpu().numpy(): move về CPU, convert sang numpy
    return batch_embeddings.cpu().numpy()

In [19]:
def get_embeddings(texts, model, tokenizer, device, batch_size=32):
    embeddings = []

    # Lặp theo batch: range(0, 5574, 32) = 0, 32, 64, ..., 5568
    for i in tqdm(range(0, len(texts), batch_size),
                   desc="Generating embeddings"):

        batch_texts = texts[i : i + batch_size]  # slice 32 texts

        # QUAN TRỌNG: thêm prefix "passage: " cho training data
        batch_texts_with_prefix = [
            f"passage: {text}" for text in batch_texts
        ]

        batch_embeddings = encode_text(
            batch_texts_with_prefix, model, tokenizer, device
        )
        embeddings.append(batch_embeddings)  # [32, 768]

    # np.vstack: ghép nối tất cả batches: [5574, 768]
    return np.vstack(embeddings)

In [20]:
# Encode labels: 'ham' → 0, 'spam' → 1
le = LabelEncoder()
y = le.fit_transform(labels)
print(f'Classes: {le.classes_}')

Classes: ['ham' 'spam']


In [21]:
# → Classes: ['ham' 'spam']
# Tạo toàn bộ embeddings: shape [5574, 768]
X_embeddings = get_embeddings(messages, model, tokenizer, device)
print(f"Embeddings shape: {X_embeddings.shape}")

Generating embeddings: 100%|██████████| 175/175 [00:22<00:00,  7.78it/s]

Embeddings shape: (5572, 768)


In [23]:
# → Embeddings shape: (5574, 768)
# Metadata: lưu thông tin kèm theo mỗi vector
metadata = []
for i, (message, label) in enumerate(zip(messages, labels)):
  metadata.append({ 'index': i, 'message': message, 'label': label, 'label_encoded': y[i] })

In [24]:
TEST_SIZE = 0.1 # 10% test
SEED = 42 # reproducibility
# stratify=y: đảm bảo tỷ lệ spam/ham như nhau ở train và test
train_indices, test_indices = train_test_split( range(len(messages)), test_size=TEST_SIZE, stratify=y, random_state=SEED ) # train: 90% = ~5016 samples  # test: 10% = ~558 samples

In [25]:
# Slice embeddings và metadata theo indice
X_train_emb = X_embeddings[train_indices] # [5016, 768]
X_test_emb = X_embeddings[test_indices] # [558, 768]

y_train = y[train_indices]
y_test = y[test_indices]

train_metadata = [metadata[i] for i in train_indices]
test_metadata = [metadata[i] for i in test_indices]

In [26]:
# Chiều của embedding vectors
embedding_dim = X_train_emb.shape[1] # 768

# IndexFlatIP:
# - "Flat" = lưu toàn bộ vectors (exact search)
# - "IP" = Inner Product (= cosine vì đã normalize)
index = faiss.IndexFlatIP(embedding_dim)

# Add vectors — PHẢI là float32!
index.add(X_train_emb.astype('float32'))

print(f"FAISS index: {index.ntotal} vectors") # → FAISS index: 5014 vectors

FAISS index: 5014 vectors


In [27]:
def classify_with_knn(query_text, model, tokenizer, device, index, train_metadata, k=1):
  # BƯỚC 1: Encode query với prefix "query: " (khác "passage: ")
  query_with_prefix = f"query: {query_text}"
  query_embedding = encode_text( [query_with_prefix], model, tokenizer, device ) # query_embedding shape: [1, 768]

  # BƯỚC 2: FAISS search → K hàng xóm gần nhất
  scores, indices = index.search(query_embedding, k)
  # scores: [[0.91, 0.87, 0.82]] — similarity cao hơn = giống hơn
  # indices: [[247, 891, 1203]] — vị trí trong train_metadata

  # BƯỚC 3: Thu thập labels của K hàng xóm
  predictions = []
  neighbor_info = []

  for i in range(k):
    neighbor_idx = indices[0][i]
    neighbor_score = scores[0][i]
    neighbor_label = train_metadata[neighbor_idx]['label']
    neighbor_msg = train_metadata[neighbor_idx]['message']

    predictions.append(neighbor_label)
    neighbor_info.append({ 'score': float(neighbor_score), 'label': neighbor_label, 'message': neighbor_msg[:100] + "..." if len(neighbor_msg) > 100 else neighbor_msg })

    # BƯỚC 4: Majority voting
    unique_labels, counts = np.unique(predictions, return_counts=True)
    final_prediction = unique_labels[np.argmax(counts)] # argmax lấy index của label có count cao nhất

    return final_prediction, neighbor_info

In [29]:
def evaluate_knn_accuracy(test_embeddings, test_labels, test_metadata, index, train_metadata, k_values=[1,3,5]):
  results = {}
  all_errors = {}

  for k in k_values:
    correct = 0
    errors = []
    total = len(test_embeddings)

    for i in tqdm(range(total), desc=f"Evaluating k={k}"): # Lấy embedding của test sample thứ i
      query_embedding = test_embeddings[i:i+1].astype('float32')
      true_label = test_metadata[i]['label']

      # FAISS search
      scores, indices = index.search(query_embedding, k)
      # Vote
      predictions = [ train_metadata[indices[0][j]]['label'] for j in range(k) ]
      unique_labels, counts = np.unique(predictions, return_counts=True)
      predicted_label = unique_labels[np.argmax(counts)]
      if predicted_label == true_label:
        correct += 1
      else:
        # Ghi lại lỗi để phân tích
        errors.append({ 'message': true_label, 'true_label': true_label, 'predicted_label': predicted_label, 'neighbors': [...] # chi tiết hàng xóm
                       })

      accuracy = correct / total
      results[k] = accuracy
      all_errors[k] = errors

    print(f"Accuracy k={k}: {accuracy:.4f}")

  return results, all_errors

In [30]:
def spam_classifier_pipeline(user_input, k=3):
  print(f"\n*** Classifying: '{user_input}'")
  print(f"*** Using top-{k} nearest neighbors\n")

  # Gọi classify function đã implement ở bước 4
  prediction, neighbors = classify_with_knn( user_input, model, tokenizer, device, index, train_metadata, k=k )
  print(f"*** Prediction: {prediction.upper()}")
  print("\n*** Top neighbors:")

  for i, neighbor in enumerate(neighbors, 1):
    print(f"{i}. Label: {neighbor['label']} | Score: {neighbor['score']:.4f}")
    print(f" Message: {neighbor['message']}\n")
    labels_list = [n['label'] for n in neighbors]
    label_counts = {l: labels_list.count(l) for l in set(labels_list)}

  return { 'prediction': prediction, 'neighbors': neighbors, 'label_distribution': label_counts }

In [32]:
# Test 1: Tin nhắn cá nhân bình thường
print("Test 1: Tin nhắn cá nhân bình thường")
result = spam_classifier_pipeline( "I am actually thinking a way of doing something useful", k=3 ) # → HAM ✅
# Top neighbor: "Ok lar... Joking wif u oni..." (ham, score ~0.82)

# Test 2: Hẹn gặp
print("Test 2: Hẹn gặp")
result = spam_classifier_pipeline( "Hey, can you pick me up at 5pm today?", k=3 ) # → HAM ✅

# Test 3: Cảm ơn sau meeting
print("Test 3: Cảm ơn sau meeting")
result = spam_classifier_pipeline( "Thanks for the meeting today, let's schedule for next week", k=3 ) # → HAM ✅

Test 1: Tin nhắn cá nhân bình thường

*** Classifying: 'I am actually thinking a way of doing something useful'
*** Using top-3 nearest neighbors

*** Prediction: HAM

*** Top neighbors:
1. Label: ham | Score: 0.8424
 Message: yeah, that's what I was thinking

Test 2: Hẹn gặp

*** Classifying: 'Hey, can you pick me up at 5pm today?'
*** Using top-3 nearest neighbors

*** Prediction: HAM

*** Top neighbors:
1. Label: ham | Score: 0.8704
 Message: Then ü come n pick me at 530 ar?

Test 3: Cảm ơn sau meeting

*** Classifying: 'Thanks for the meeting today, let's schedule for next week'
*** Using top-3 nearest neighbors

*** Prediction: HAM

*** Top neighbors:
1. Label: ham | Score: 0.8398
 Message: Lets use it next week, princess :)

